# 🧪 Lab 2 — Praktikum Matematika untuk ML

> Pendamping materi Bab 2. Prinsip: **matematika dipahami lewat kode + visual, bukan hafalan rumus**.

| Bagian | Topik | Waktu (menit) |
|---|---|---|
| 1 | Vektor, dot product, kemiripan | 20 |
| 2 | Cosine similarity — dasar RAG | 20 |
| 3 | Turunan & gradien | 15 |
| 4 | Gradient descent + learning rate | 25 |
| 5 | Perkalian matriks | 15 |
| 6 | Statistik deskriptif | 15 |

Setiap latihan punya cell **✅ Cek** — jalankan, kalau ✅ berarti pemahamanmu benar.

## Bagian 1 — Vektor & Dot Product

Vektor = daftar angka yang merepresentasikan satu data point. Dot product = ukuran "kesesuaian arah".

In [ ]:
import numpy as np

a = np.array([1.0, 2.0, 3.0])
b = np.array([4.0, 5.0, 6.0])

print('a · b manual :', sum(x*y for x, y in zip(a, b)))   # definisi: jumlah hasil kali
print('a · b numpy  :', np.dot(a, b))                     # cara cepat
print('norm(a)      :', np.linalg.norm(a))                # panjang vektor = sqrt(1+4+9)
print('norm(b)      :', np.linalg.norm(b))

# Dot product vs panjang: vektor yang SAMA arah tapi beda panjang
print('a·a          :', np.dot(a, a))              # = norm(a)^2
print('a·(2a)       :', np.dot(a, 2*a))            # 2x lebih besar — panjang mempengaruhi!

**Observasi penting:** dot product dipengaruhi PANJANG vektor. Vektor panjang yang agak beda arah bisa "skor"-nya lebih tinggi dari vektor pendek yang arahnya sama persis. Solusinya → normalisasi → cosine similarity (Bagian 2).

### 🎯 Latihan 1.1 — Jarak Euclidean

Jarak antar dua data point = panjang vektor selisihnya: `d(a, b) = ||a - b||₂`.

In [ ]:
def jarak_euclidean(p, q):
    """Return jarak Euclidean antara dua vektor numpy (tanpa loop Python)."""
    # TODO: implementasikan (hint: norm dari selisih)
    raise NotImplementedError

# --- uji sendiri ---
# p, q = np.array([0., 0.]), np.array([3., 4.])
# print(jarak_euclidean(p, q))   # harusnya 5.0

In [ ]:
# ✅ Cek latihan 1.1
p, q = np.array([0., 0.]), np.array([3., 4.])
assert abs(jarak_euclidean(p, q) - 5.0) < 1e-9, '3-4-5 triangle'
assert abs(jarak_euclidean(q, q)) < 1e-9, 'jarak ke diri sendiri = 0'
assert jarak_euclidean(p, q) == jarak_euclidean(q, p), 'simetris'
print('✅ Latihan 1.1 benar!')

## Bagian 2 — Cosine Similarity: Mesin di Balik RAG

Cosine similarity mengukur **sudut** antar vektor, mengabaikan panjang:

`cos(θ) = (a · b) / (||a|| ||b||)`

- `1` = arah sama persis · `0` = tegak lurus (tak berhubungan) · `-1` = berlawanan.

In [ ]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# "Embedding" buatan: 3 dimensi = [topik_kopi, topik_teh, topik_mesin]
doc_kopi   = np.array([0.9, 0.1, 0.0])
doc_kopi2  = np.array([0.8, 0.2, 0.1])
doc_teh    = np.array([0.1, 0.9, 0.0])
doc_mesin  = np.array([0.0, 0.0, 1.0])

docs = {'kopi': doc_kopi, 'kopi2': doc_kopi2, 'teh': doc_teh, 'mesin': doc_mesin}
names = list(docs)
for i in range(len(names)):
    for j in range(i+1, len(names)):
        s = cosine_similarity(docs[names[i]], docs[names[j]])
        print(f'{names[i]:>6} vs {names[j]:<6}: {s:+.3f}')

### 🎯 Latihan 2.1 — Mini Search Engine (inti RAG!)

Bangun fungsi pencarian dokumen paling relevan untuk sebuah query — **persis** cara kerja retrieval di RAG (Bab 12), hanya embeddinger-nya versi manual.

In [ ]:
korpora = {
    'cara_membuat_kopi':   np.array([0.9, 0.1, 0.0, 0.0]),
    'jenis_teh_hijau':     np.array([0.1, 0.9, 0.0, 0.0]),
    'mesin_espresso':      np.array([0.7, 0.0, 0.7, 0.0]),
    'servis_mesin_mobil':  np.array([0.0, 0.0, 0.9, 0.1]),
    'resep_kue_coklat':    np.array([0.0, 0.2, 0.0, 0.9]),
}

def cari_dokumen(korpora: dict, query: np.ndarray, k: int = 2) -> list[str]:
    """Return nama-nama dokumen dengan cosine similarity TERTINGGI terhadap query.
    Urut menurun, sebanyak k buah.
    """
    # TODO: hitung similarity query ke semua dokumen, ambil top-k namanya
    raise NotImplementedError

# --- uji sendiri ---
# q_kopi = np.array([0.8, 0.1, 0.1, 0.0])
# print(cari_dokumen(korpora, q_kopi, k=2))
# q_mesin = np.array([0.0, 0.0, 1.0, 0.0])
# print(cari_dokumen(korpora, q_mesin, k=2))

In [ ]:
# ✅ Cek latihan 2.1
q_kopi = np.array([0.8, 0.1, 0.1, 0.0])
q_mesin = np.array([0.0, 0.0, 1.0, 0.0])
hasil1 = cari_dokumen(korpora, q_kopi, k=2)
hasil2 = cari_dokumen(korpora, q_mesin, k=2)
assert hasil1[0] == 'cara_membuat_kopi', f'top-1 untuk query kopi salah: {hasil1}'
assert 'mesin_espresso' in hasil1, 'espresso harus relevan dengan kopi'
assert hasil2[0] in ('servis_mesin_mobil', 'mesin_espresso'), f'top-1 untuk query mesin salah: {hasil2}'
assert len(hasil1) == 2 and len(hasil2) == 2
print('✅ Latihan 2.1 benar! Kamu baru saja membangun inti retrieval RAG.')

## Bagian 3 — Turunan & Gradien

Turunan = laju perubahan. Gradien = vektor turunan parsial — arah **naik** paling curam.

In [ ]:
import numpy as np

# Turunan numerik: (f(x+h) - f(x-h)) / 2h  — cek turunan analitik kamu
def turunan_numerik(f, x, h=1e-5):
    return (f(x + h) - f(x - h)) / (2 * h)

f  = lambda x: x ** 2
df = lambda x: 2 * x          # turunan analitik

for x in [-2, -0.5, 0, 1, 3]:
    print(f'x={x:>4}: analitik={df(x):>5}  numerik={turunan_numerik(f, x):.5f}')

# Gradien fungsi 2 variabel
g  = lambda p: p[0]**2 + p[1]**2            # f(x, y) = x² + y²
dg = lambda p: np.array([2*p[0], 2*p[1]])   # ∇f = [2x, 2y]

titik = np.array([1.0, 2.0])
print('gradien di (1,2):', dg(titik), '-> arah naik tercepat')
print('negatif gradien :', -dg(titik), '-> arah TURUN tercepat (yang dipakai GD)')

## Bagian 4 — Gradient Descent

Algoritma yang melatih hampir semua model ML:

```
ulangi:  w = w - learning_rate × gradien(w)
```

In [ ]:
import matplotlib.pyplot as plt

def loss(w):
    return (w - 3) ** 2                 # minimum di w = 3

def gradient(w):
    return 2 * (w - 3)                  # d/dw (w-3)²

def gradient_descent(w0, lr, n_steps=20):
    jalur = [w0]
    for _ in range(n_steps):
        w = jalur[-1]
        jalur.append(w - lr * gradient(w))
    return np.array(jalur)

# Bandingkan 3 learning rate
for lr in [0.01, 0.2, 1.05]:
    jalur = gradient_descent(0.0, lr)
    akhir = jalur[-1]
    print(f'lr={lr:<5} -> w_akhir={akhir:>12.4f}  loss_akhir={loss(akhir):.6f}')

In [ ]:
# Visualisasi ketiga jalur
ws = np.linspace(-1, 7, 300)
plt.figure(figsize=(10, 6))
plt.plot(ws, loss(ws), 'k-', alpha=0.4, label='J(w) = (w-3)²')
for lr, warna in [(0.01, 'blue'), (0.2, 'green'), (1.05, 'red')]:
    jalur = gradient_descent(0.0, lr)
    plt.plot(jalur, loss(jalur), 'o--', color=warna, alpha=0.7, label=f'lr={lr}')
plt.axvline(3, color='gray', ls=':', label='minimum w=3')
plt.xlabel('w'); plt.ylabel('J(w)'); plt.title('Efek Learning Rate')
plt.legend(); plt.grid(alpha=0.3); plt.show()

### 🎯 Latihan 4.1 — Generic Gradient Descent

Buat GD untuk fungsi loss **apa pun** — ini pola yang sama dipakai saat melatih neural network (Bab 4).

In [ ]:
def gradient_descent_generic(loss_fn, grad_fn, w0, lr=0.1, n_steps=50, tol=1e-6):
    """Jalankan GD dari w0.
    Return (w_akhir, list_loss_per_step).
    Berhenti lebih awal jika perubahan |w_baru - w_lama| < tol.
    """
    # TODO: implementasikan loop GD + early stop
    raise NotImplementedError

# --- uji sendiri ---
# w, riwayat = gradient_descent_generic(loss, gradient, w0=0.0, lr=0.2)
# print(f'w={w:.6f} setelah {len(riwayat)} evaluasi loss')

In [ ]:
# ✅ Cek latihan 4.1
w, riwayat = gradient_descent_generic(loss, gradient, w0=0.0, lr=0.2)
assert abs(w - 3.0) < 1e-4, f'harus konvergen ke 3, dapat {w}'
assert len(riwayat) >= 2 and riwayat[0] > riwayat[-1], 'loss harus menurun'
assert all(riwayat[i] >= riwayat[i+1] - 1e-12 for i in range(len(riwayat)-1)), 'menurun monoton (lr=0.2 aman)'

# Early stop: lr sangat kecil + tol longgar -> berhenti sebelum n_steps
w2, r2 = gradient_descent_generic(loss, gradient, w0=0.0, lr=0.001, n_steps=10_000, tol=1e-3)
assert len(r2) < 10_000, 'early stop harus aktif'
print('✅ Latihan 4.1 benar!')

## Bagian 5 — Perkalian Matriks = Kombinasi Linear

`A @ v` = gabungan kolom-kolom A, di-scale oleh komponen v. Inilah cara melihat layer neural network.

In [ ]:
A = np.array([
    [1., 2., 3.],
    [4., 5., 6.],
])
v = np.array([1., 0., -1.])

hasil_manual = v[0]*A[:, 0] + v[1]*A[:, 1] + v[2]*A[:, 2]
hasil_numpy  = A @ v
print('manual:', hasil_manual)
print('numpy :', hasil_numpy)
print('sama? ', np.allclose(hasil_manual, hasil_numpy))

# Bentuk: (m, n) @ (n,) -> (m,). Dan (m, n) @ (n, k) -> (m, k)
B = np.array([[1., 0.], [0., 1.], [1., 1.]])
print('A@B shape:', (A @ B).shape)   # (2, 3) @ (3, 2) -> (2, 2)

### 🎯 Latihan 5.1 — Neural Layer Mini

Satu layer neural network = `output = aktivasi(X @ W + b)`. Bangun dari nol!

In [ ]:
def relu(z):
    """ReLU: max(0, z), tanpa loop."""
    # TODO
    raise NotImplementedError

def layer_forward(X, W, b):
    """X: (n_sampel, n_fitur), W: (n_fitur, n_unit), b: (n_unit,).
    Return aktivasi ReLU dari X @ W + b.
    """
    # TODO
    raise NotImplementedError

X = np.array([[1., 2.], [3., -1.], [0., 0.]])   # 3 sampel, 2 fitur
W = np.array([[0.5, -1.], [0.25, 2.]])          # 2 fitur, 2 unit
b = np.array([0.1, -0.1])

# out = layer_forward(X, W, b)
# print(out)

In [ ]:
# ✅ Cek latihan 5.1
assert relu(np.array([-2., 0., 3.])).tolist() == [0.0, 0.0, 3.0], 'relu dasar'
out = layer_forward(X, W, b)
expected = np.maximum(X @ W + b, 0)
assert np.allclose(out, expected), f'harus {expected.tolist()}, dapat {out.tolist()}'
assert out.shape == (3, 2)
print('✅ Latihan 5.1 benar! Kamu baru menghitung forward pass layer NN pertamamu.')

## Bagian 6 — Statistik Deskriptif

In [ ]:
data = np.array([85, 90, 78, 92, 88, 60, 95])

print('mean :', np.mean(data))
print('std  :', np.std(data))               # seberapa tersebar
print('median:', np.median(data))
print('min/max:', data.min(), data.max())

# Aturan 68-95: dari 7 nilai, berapa yang dalam mean ± 2std?
m, s = np.mean(data), np.std(data)
dalam_2std = np.mean(np.abs(data - m) <= 2 * s)
print(f'dalam ±2std: {dalam_2std:.0%}')

# Korelasi: hubungan linear dua variabel
jam_belajar = np.array([1, 2, 3, 4, 5, 6, 7])
nilai       = np.array([60, 65, 70, 75, 78, 85, 90])
r = np.corrcoef(jam_belajar, nilai)[0, 1]
print(f'korelasi jam_belajar vs nilai: {r:.3f}')
print('-> korelasi tinggi TIDAK berarti jam belajar MENYEBABKAN nilai tinggi!')

### 🎯 Latihan 6.1 — Z-score & Outlier

Z-score = berapa std jauhnya sebuah nilai dari mean. Standar untuk normalisasi & deteksi outlier.

In [ ]:
def zscore(data):
    """Return z-score tiap elemen: (x - mean) / std. Tanpa loop."""
    # TODO
    raise NotImplementedError

def deteksi_outlier(data, ambang=2.0):
    """Return boolean array: True jika |z| > ambang."""
    # TODO
    raise NotImplementedError

penjualan = np.array([120, 135, 128, 122, 131, 400, 125, 119])   # 400 mencurigakan

# print(zscore(penjualan).round(2))
# print(deteksi_outlier(penjualan, 2.0))

In [ ]:
# ✅ Cek latihan 6.1
z = zscore(penjualan)
assert abs(z.mean()) < 1e-9, 'z-score selalu mean 0'
assert abs(z.std() - 1.0) < 1e-9, 'z-score selalu std 1'
out = deteksi_outlier(penjualan, 2.0)
assert out.tolist()[5] == True, '400 harus terdeteksi outlier'
assert out.sum() == 1, f'cuma 1 outlier, dapat {out.sum()}'
print('✅ Latihan 6.1 benar!')

## 🏁 Penutup Lab

**Refleksi:**
1. Jelaskan gradient descent ke anak SMA dalam 3 kalimat — tanpa kata "turunan" atau "gradien".
2. Kapan cosine similarity lebih tepat daripada jarak Euclidean? (hint: bagian 1 — pengaruh panjang vektor)
3. Mana yang lebih menarik: bagian 2 (search engine) atau bagian 4 (melatih model)? Itu petunjuk minat bab lanjutanmu.

**Lanjut:** `02_kuis_matematika.ipynb` → `project-starter-gradient-descent/` → `cheatsheet-matematika-ml.md`.